In [109]:
from google.cloud import storage
import os
from dotenv import load_dotenv
import pandas as pd
from io import BytesIO
import numpy as np

load_dotenv()

True

In [110]:
GCS_BUCKET = os.getenv("GCS_BUCKET", "").strip()
GCS_MEDAL_FETCH = os.getenv("GCS_MEDAL_FETCH", "").strip()
BLOB_FETCH_PATH = f"{GCS_MEDAL_FETCH}/season=2026/session_result.parquet"

BLOB_FETCH_SESSIONS = f"gold/season=2026/session.parquet"
BLOB_FETCH_DRIVERS = f"gold/season=2026/driver.parquet"

In [111]:
def fill_position_by_laps(group):
    g = group.copy()
    pos = g["position"]
    known = pos.notna()
    if known.all():
        return g

    last = int(pos[known].max()) if known.any() else 0
    miss = g.loc[~known].copy()
    miss = miss.sort_values(
        by=["number_of_laps", "driver_sort"],
        ascending=[False, True],
        kind="mergesort",
    )

    miss["position"] = np.arange(1, len(miss) + 1, dtype="int64") + last
    g.loc[miss.index, "position"] = miss["position"].astype("Int64")
    return g

In [112]:
client = storage.Client()
bucket = client.bucket(GCS_BUCKET)
blob = bucket.blob(BLOB_FETCH_PATH)
df = pd.read_parquet(BytesIO(blob.download_as_bytes()))

blob_sessions = bucket.blob(BLOB_FETCH_SESSIONS)
df_sessions = pd.read_parquet(BytesIO(blob_sessions.download_as_bytes()))

blob_drivers = bucket.blob(BLOB_FETCH_DRIVERS)
df_drivers = pd.read_parquet(BytesIO(blob_drivers.download_as_bytes()))


In [113]:
good_keys = df_sessions.loc[
    df_sessions["session_name"].isin(["Sprint", "Race"])
    & (~df_sessions["is_cancelled"]),
    "session_key",
]

df = df.loc[df["session_key"].isin(good_keys)]

In [114]:
df = df.sort_values(
    by=["session_key", "points"],
    ascending=[True, False]
)

df = df.drop_duplicates(subset=["session_key", "driver_number"])
df["driver_sort"] = df["full_name"].astype(str)   # atau kolom lain

In [116]:
column_convert = ["number_of_laps", "points", "position"]
df["number_of_laps"] = df["number_of_laps"].fillna(0).astype(int)
df["points"] = df["points"].astype(int)
df = df.groupby("session_key", group_keys=False).apply(fill_position_by_laps)

C:\Users\USER\AppData\Local\Temp\ipykernel_44396\4012158921.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("session_key", group_keys=False).apply(fill_position_by_laps)


In [117]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 132 entries, 0 to 131
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   meeting_key     132 non-null    int64              
 1   session_key     132 non-null    int64              
 2   session_name    132 non-null    object             
 3   location        132 non-null    object             
 4   driver_number   132 non-null    int64              
 5   full_name       132 non-null    object             
 6   country         132 non-null    object             
 7   team_name       132 non-null    object             
 8   number_of_laps  132 non-null    int64              
 9   dnf             132 non-null    bool               
 10  dns             132 non-null    bool               
 11  dsq             132 non-null    bool               
 12  gap_to_leader   108 non-null    string             
 13  duration        79 non-null     string  

In [118]:
df.head()

,meeting_key,session_key,session_name,location,driver_number,full_name,country,team_name,number_of_laps,dnf,dns,dsq,gap_to_leader,duration,points,position,ingested_at,driver_sort
0,1279,11234,Race,Melbourne,63,George RUSSELL,United Kingdom,Mercedes,58,False,False,False,0,4986.801,25,1.0,2026-05-04 19:22:08.656357+00:00,George RUSSELL
4,1279,11234,Race,Melbourne,12,Kimi ANTONELLI,Italy,Mercedes,58,False,False,False,2.974,4989.775,18,2.0,2026-05-04 19:22:08.656357+00:00,Kimi ANTONELLI
8,1279,11234,Race,Melbourne,16,Charles LECLERC,Monaco,Ferrari,58,False,False,False,15.519,5002.32,15,3.0,2026-05-04 19:22:08.656357+00:00,Charles LECLERC
12,1279,11234,Race,Melbourne,44,Lewis HAMILTON,United Kingdom,Ferrari,58,False,False,False,16.144,5002.945,12,4.0,2026-05-04 19:22:08.656357+00:00,Lewis HAMILTON
16,1279,11234,Race,Melbourne,1,Lando NORRIS,United Kingdom,McLaren,58,False,False,False,51.741,5038.542,10,5.0,2026-05-04 19:22:08.656357+00:00,Lando NORRIS


In [119]:
df_drivers = df_drivers[["driver_number", "full_name", "country", "team_name"]]
df_drivers["TotalPoints"] = 0

In [120]:
totals = (
    df.groupby("driver_number", as_index=False)["points"]
    .sum()
    .rename(columns={"points": "TotalPoints"})
)

df_drivers = df_drivers.drop(columns=["TotalPoints"], errors="ignore")
df_drivers = df_drivers.merge(totals, on="driver_number", how="left")
df_drivers["TotalPoints"] = df_drivers["TotalPoints"].fillna(0).astype(int)

df_drivers = df_drivers.sort_values(by="TotalPoints", ascending=False)

In [129]:
team_points = (
    df_drivers.groupby("team_name", as_index=False)["TotalPoints"]
    .sum()
)
team_points = team_points.sort_values(by="TotalPoints", ascending=False)
team_points


,team_name,TotalPoints
7,Mercedes,180
4,Ferrari,110
6,McLaren,94
9,Red Bull Racing,30
0,Alpine,23
5,Haas F1 Team,18
8,Racing Bulls,14
10,Williams,5
2,Audi,2
1,Aston Martin,0
